# 83: MVRV + Momentum Deep Dive (The Winning Strategy)

**🎉 BREAKTHROUGH from Notebook 82:**

```
Never Exit:              844.9%
MVRV>2.0 AND Price<50MA: 2784.1%  ← 3.3x BETTER!
```

After testing 80+ strategies across 9 notebooks, we finally found one that **CRUSHES** "Never Exit"!

## Why This Works:

**MVRV > 2.0** (Valuation)
- Market is historically expensive (top 10% of valuations)
- Realized value significantly below market price
- High risk of mean reversion

**Price < 50-day MA** (Momentum)
- Uptrend has broken
- Price action confirming valuation concern
- 50-day MA is the "institutional trend" line

**Combined:**
- Stay invested when expensive but trending up (2023-2026)
- Exit when expensive AND trend breaks (2017/2021 tops)

## What This Notebook Does:

1. **Trade-by-trade analysis** - When did exits fire? What happened next?
2. **Parameter optimization** - Is MVRV 2.0 optimal? What about 1.8, 2.2, 2.5?
3. **MA period testing** - Is 50-day best? What about 30, 75, 100-day?
4. **Historical validation** - Does it work in 2013, 2017, 2021 tops?
5. **Risk analysis** - What's the worst drawdown? Win rate? Time in market?
6. **Robustness check** - Is this a fluke or a real edge?

**Goal:** Validate this as a production-ready strategy or find what makes it tick.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
df.head()

## 2. Calculate Indicators

In [ ]:
print("Calculating indicators...\n")

price = df['price']

# Moving averages (test multiple periods)
for period in [20, 30, 50, 75, 100, 200]:
    df[f'ma_{period}'] = price.rolling(period).mean()
    df[f'above_ma_{period}'] = price > df[f'ma_{period}']

print("✓ Indicators calculated")
print(f"\nCurrent state:")
print(f"  Price: ${price.iloc[-1]:,.0f}")
print(f"  MVRV: {df['mvrv'].iloc[-1]:.2f}")
print(f"  50-day MA: ${df['ma_50'].iloc[-1]:,.0f} ({'above' if df['above_ma_50'].iloc[-1] else 'below'})")
print(f"  100-day MA: ${df['ma_100'].iloc[-1]:,.0f} ({'above' if df['above_ma_100'].iloc[-1] else 'below'})")

## 3. Generate Entry Signals

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
print("Generating entry signals...\n")

c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0

c4 = c4.fillna(False)
c5 = c5.fillna(False)

entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = (entry_count >= 4).fillna(False).astype(bool)

print(f"✓ Entry signals: {entries.sum()}")

## 4. Parameter Grid Search: MVRV Threshold x MA Period

In [ ]:
print("Running parameter grid search...\n")

# Test MVRV thresholds
mvrv_thresholds = [1.5, 1.8, 2.0, 2.2, 2.5, 3.0]

# Test MA periods
ma_periods = [20, 30, 50, 75, 100]

# Store results
grid_results = []

def backtest_strategy(df, entries, exits, name):
    """Quick backtest helper."""
    try:
        pf = vbt.Portfolio.from_signals(
            close=df['price'],
            entries=entries,
            exits=exits,
            fees=FEES,
            slippage=SLIPPAGE,
            init_cash=10000,
            freq='1D'
        )
        return {
            'name': name,
            'total_return': pf.total_return() * 100,
            'sharpe': pf.sharpe_ratio(),
            'max_dd': pf.max_drawdown() * 100,
            'num_trades': pf.trades.count(),
            'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
        }
    except:
        return None

# Baseline: Never Exit
never_exits = pd.Series(False, index=df.index, dtype=bool)
never_result = backtest_strategy(df, entries, never_exits, 'Never Exit')
print(f"Baseline (Never Exit): {never_result['total_return']:.1f}%\n")

# Grid search
print("Testing parameter combinations...")
for mvrv_thresh in mvrv_thresholds:
    for ma_period in ma_periods:
        # Generate exit signal
        mvrv_condition = (df['mvrv'] > mvrv_thresh).fillna(False)
        ma_condition = (~df[f'above_ma_{ma_period}']).fillna(False)
        exits = (mvrv_condition & ma_condition).astype(bool)
        
        name = f"MVRV>{mvrv_thresh} + Price<{ma_period}MA"
        result = backtest_strategy(df, entries, exits, name)
        
        if result:
            result['mvrv_threshold'] = mvrv_thresh
            result['ma_period'] = ma_period
            result['exit_signals'] = exits.sum()
            grid_results.append(result)

# Convert to DataFrame
results_df = pd.DataFrame(grid_results)
results_df = results_df.sort_values('total_return', ascending=False)

print(f"\n✓ Tested {len(grid_results)} combinations")
print(f"\nTop 10 Strategies:")
print("="*100)
print(f"{'Strategy':<40} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Exits':>8}")
print("-"*100)
for _, row in results_df.head(10).iterrows():
    print(f"{row['name']:<40} {row['total_return']:>9.1f}% {row['sharpe']:>8.2f} {row['max_dd']:>7.1f}% {int(row['num_trades']):>8} {int(row['exit_signals']):>8}")
print("="*100)

## 5. Heatmap: Performance by Parameters

In [ ]:
# Create heatmap of returns
pivot_returns = results_df.pivot(index='mvrv_threshold', columns='ma_period', values='total_return')
pivot_sharpe = results_df.pivot(index='mvrv_threshold', columns='ma_period', values='sharpe')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Returns heatmap
ax1 = axes[0]
sns.heatmap(pivot_returns, annot=True, fmt='.0f', cmap='RdYlGn', center=never_result['total_return'],
            ax=ax1, cbar_kws={'label': 'Total Return (%)'})
ax1.set_title('Total Return by Parameters', fontsize=14, fontweight='bold')
ax1.set_xlabel('MA Period (days)', fontsize=12)
ax1.set_ylabel('MVRV Threshold', fontsize=12)

# Sharpe heatmap
ax2 = axes[1]
sns.heatmap(pivot_sharpe, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5,
            ax=ax2, cbar_kws={'label': 'Sharpe Ratio'})
ax2.set_title('Sharpe Ratio by Parameters', fontsize=14, fontweight='bold')
ax2.set_xlabel('MA Period (days)', fontsize=12)
ax2.set_ylabel('MVRV Threshold', fontsize=12)

plt.tight_layout()
plt.show()

print("\n📊 Heatmap shows parameter sensitivity")
print("   Green = Good performance")
print("   Red = Poor performance")

## 6. Deep Dive: Best Strategy Trade Analysis

In [ ]:
# Get best strategy
best = results_df.iloc[0]
best_mvrv = best['mvrv_threshold']
best_ma = int(best['ma_period'])

print(f"🏆 BEST STRATEGY: {best['name']}")
print(f"   Return: {best['total_return']:.1f}%")
print(f"   Sharpe: {best['sharpe']:.2f}")
print(f"   Max DD: {best['max_dd']:.1f}%")
print(f"   Trades: {int(best['num_trades'])}")
print(f"   Win Rate: {best['win_rate']:.1f}%")

# Recreate best strategy
mvrv_condition = (df['mvrv'] > best_mvrv).fillna(False)
ma_condition = (~df[f'above_ma_{best_ma}']).fillna(False)
best_exits = (mvrv_condition & ma_condition).astype(bool)

# Run full backtest
pf_best = vbt.Portfolio.from_signals(
    close=df['price'],
    entries=entries,
    exits=best_exits,
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

pf_never = vbt.Portfolio.from_signals(
    close=df['price'],
    entries=entries,
    exits=never_exits,
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

print(f"\n📊 COMPARISON TO NEVER EXIT:")
print(f"   Never Exit: {never_result['total_return']:.1f}%")
print(f"   Best Strategy: {best['total_return']:.1f}%")
print(f"   Improvement: +{best['total_return'] - never_result['total_return']:.1f}%")
print(f"   Multiple: {best['total_return'] / never_result['total_return']:.2f}x")

## 7. Exit Signal Analysis: When and Why?

In [ ]:
# Analyze exit signals
print("\n" + "="*90)
print("EXIT SIGNAL ANALYSIS")
print("="*90)

exit_dates = df[best_exits].index

print(f"\nTotal exit signals: {len(exit_dates)}")
print(f"\nExit signals by year:")
for year in sorted(df.index.year.unique()):
    year_exits = sum(exit_dates.year == year)
    if year_exits > 0:
        print(f"  {year}: {year_exits}")

print(f"\n📋 DETAILED EXIT ANALYSIS:")
print("="*90)

for i, exit_date in enumerate(exit_dates[:20]):  # Show first 20
    exit_price = df.loc[exit_date, 'price']
    mvrv_val = df.loc[exit_date, 'mvrv']
    ma_val = df.loc[exit_date, f'ma_{best_ma}']
    
    # Forward returns
    try:
        future_idx = df.index.get_loc(exit_date)
        if future_idx + 30 < len(df):
            future_30d = df.iloc[future_idx + 30]['price']
            ret_30d = (future_30d / exit_price - 1) * 100
        else:
            ret_30d = None
            
        if future_idx + 90 < len(df):
            future_90d = df.iloc[future_idx + 90]['price']
            ret_90d = (future_90d / exit_price - 1) * 100
        else:
            ret_90d = None
        
        print(f"\n{i+1}. {exit_date.date()}:")
        print(f"   Exit Price: ${exit_price:,.0f}")
        print(f"   MVRV: {mvrv_val:.2f} (threshold: {best_mvrv})")
        print(f"   {best_ma}MA: ${ma_val:,.0f} (price {(exit_price/ma_val - 1)*100:+.1f}% vs MA)")
        if ret_30d is not None:
            print(f"   Forward Returns: 30d: {ret_30d:+.1f}%", end="")
            if ret_90d is not None:
                print(f" | 90d: {ret_90d:+.1f}%")
            else:
                print()
    except:
        pass

if len(exit_dates) > 20:
    print(f"\n... and {len(exit_dates) - 20} more exit signals")

## 8. Equity Curves Comparison

In [ ]:
# Plot equity curves
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Equity curves
ax1 = axes[0]
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
best_equity = pf_best.value()
never_equity = pf_never.value()

ax1.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=2.5, color='gray', linestyle='--', alpha=0.7)
ax1.plot(never_equity.index, never_equity.values, label='Never Exit', linewidth=2, color='blue', alpha=0.8)
ax1.plot(best_equity.index, best_equity.values, label=f'{best["name"]}', linewidth=2.5, color='green', alpha=0.9)

# Mark exit signals
for exit_date in exit_dates:
    if exit_date in best_equity.index:
        ax1.axvline(exit_date, color='red', alpha=0.2, linewidth=1)

ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.set_title('Equity Curves: Best Strategy vs Baselines', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=11, loc='upper left')
ax1.grid(True, alpha=0.3)

# Price with MVRV zones and MA
ax2 = axes[1]
ax2.plot(df.index, df['price'], label='Price', linewidth=2, color='black')
ax2.plot(df.index, df[f'ma_{best_ma}'], label=f'{best_ma}-day MA', linewidth=1.5, color='blue', alpha=0.7)

# Shade when MVRV > threshold
ax2.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(df['mvrv'] > best_mvrv), alpha=0.2, color='orange', label=f'MVRV>{best_mvrv}')

# Mark actual exits
for exit_date in exit_dates:
    if exit_date in df.index:
        exit_price = df.loc[exit_date, 'price']
        ax2.scatter(exit_date, exit_price, color='red', s=100, marker='v', zorder=5, alpha=0.7)

ax2.set_ylabel('BTC Price ($)', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_yscale('log')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Red vertical lines = Exit signals")
print("📊 Orange shading = MVRV > threshold (expensive)")
print("📊 Red triangles = Actual exit points")

## 9. Performance by Market Cycle

In [ ]:
# Test by historical periods
periods = [
    ('2013-01-01', '2015-12-31', '2013-2015 (Early)'),
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY MARKET CYCLE")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    period_exits_best = best_exits[(best_exits.index >= start) & (best_exits.index <= end)]
    period_exits_never = never_exits[(never_exits.index >= start) & (never_exits.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test strategies
    try:
        pf_best_period = vbt.Portfolio.from_signals(
            close=period_df['price'], entries=period_entries, exits=period_exits_best,
            fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
        )
        best_ret = pf_best_period.total_return() * 100
        best_sharpe = pf_best_period.sharpe_ratio()
        best_trades = pf_best_period.trades.count()
    except:
        best_ret = 0
        best_sharpe = 0
        best_trades = 0
    
    try:
        pf_never_period = vbt.Portfolio.from_signals(
            close=period_df['price'], entries=period_entries, exits=period_exits_never,
            fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
        )
        never_ret = pf_never_period.total_return() * 100
    except:
        never_ret = 0
    
    # Exit analysis
    period_exit_count = period_exits_best.sum()
    
    print(f"\n{label}:")
    print(f"  Exit signals: {period_exit_count}")
    print(f"  Buy & Hold:   {bh:+.1f}%")
    print(f"  Never Exit:   {never_ret:+.1f}% ({never_ret - bh:+.1f}% vs B&H)")
    print(f"  Best Strategy: {best_ret:+.1f}% ({best_ret - bh:+.1f}% vs B&H)")
    print(f"  Advantage:    {best_ret - never_ret:+.1f}% vs Never Exit")
    print(f"  Sharpe:       {best_sharpe:.2f}")
    print(f"  Trades:       {best_trades}")
    
    if best_ret > never_ret:
        print(f"  ✅ Best strategy WINS this period")
    else:
        print(f"  ❌ Never Exit wins this period")

print("\n" + "="*100)

## 10. Risk Metrics Deep Dive

In [ ]:
print("\n" + "="*90)
print("RISK METRICS COMPARISON")
print("="*90)

# Drawdown analysis
best_dd = pf_best.drawdown()
never_dd = pf_never.drawdown()

print(f"\nDrawdown Statistics:")
print(f"  Best Strategy:")
print(f"    Max Drawdown: {pf_best.max_drawdown() * 100:.1f}%")
print(f"    Avg Drawdown: {best_dd.mean() * 100:.1f}%")
print(f"  Never Exit:")
print(f"    Max Drawdown: {pf_never.max_drawdown() * 100:.1f}%")
print(f"    Avg Drawdown: {never_dd.mean() * 100:.1f}%")

# Trade statistics
print(f"\nTrade Statistics (Best Strategy):")
if pf_best.trades.count() > 0:
    print(f"  Total Trades: {pf_best.trades.count()}")
    print(f"  Win Rate: {pf_best.trades.win_rate() * 100:.1f}%")
    if pf_best.trades.winning.count() > 0:
        print(f"  Avg Win: {pf_best.trades.winning.returns.mean() * 100:.1f}%")
    if pf_best.trades.losing.count() > 0:
        print(f"  Avg Loss: {pf_best.trades.losing.returns.mean() * 100:.1f}%")
    print(f"  Profit Factor: {pf_best.trades.profit_factor():.2f}")
else:
    print(f"  No trades completed")

# Time in market (using positions mask instead of records)
try:
    best_positions_mask = pf_best.positions.mask
    best_time_in_market = best_positions_mask.sum() / len(df) * 100
    print(f"\nTime in Market:")
    print(f"  Best Strategy: {best_time_in_market:.1f}%")
    print(f"  In Cash: {100 - best_time_in_market:.1f}%")
except:
    # Alternative calculation
    try:
        best_holdings = pf_best.holdings()
        best_time_in_market = (best_holdings > 0).sum() / len(df) * 100
        print(f"\nTime in Market:")
        print(f"  Best Strategy: {best_time_in_market:.1f}%")
        print(f"  In Cash: {100 - best_time_in_market:.1f}%")
    except:
        print(f"\nTime in Market: Unable to calculate")

# Returns distribution
print(f"\nReturns Distribution:")
best_returns = pf_best.returns()
print(f"  Mean Daily Return: {best_returns.mean() * 100:.3f}%")
print(f"  Std Daily Return: {best_returns.std() * 100:.3f}%")
print(f"  Sharpe Ratio: {pf_best.sharpe_ratio():.2f}")
print(f"  Sortino Ratio: {pf_best.sortino_ratio():.2f}")
print(f"  Calmar Ratio: {pf_best.calmar_ratio():.2f}")

## 11. Final Verdict & Recommendations

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: IS THIS STRATEGY PRODUCTION-READY?")
print("="*90)

best = results_df.iloc[0]
improvement = best['total_return'] - never_result['total_return']
multiplier = best['total_return'] / never_result['total_return']

print(f"\n🏆 BEST STRATEGY: {best['name']}")
print(f"\n1. ABSOLUTE PERFORMANCE:")
print(f"   Return: {best['total_return']:.1f}%")
print(f"   vs Never Exit: +{improvement:.1f}% ({multiplier:.2f}x better)")
print(f"   Sharpe: {best['sharpe']:.2f}")
print(f"   Max DD: {best['max_dd']:.1f}%")

print(f"\n2. PARAMETER ROBUSTNESS:")
# Check how many combinations beat Never Exit
better_than_never = (results_df['total_return'] > never_result['total_return']).sum()
total_combos = len(results_df)
print(f"   Combinations tested: {total_combos}")
print(f"   Beat 'Never Exit': {better_than_never} ({better_than_never/total_combos*100:.1f}%)")

if better_than_never / total_combos > 0.5:
    print(f"   ✅ Strategy is ROBUST (>50% of params work)")
elif better_than_never / total_combos > 0.3:
    print(f"   ✓ Strategy is moderately robust (30-50% of params work)")
else:
    print(f"   ⚠️  Strategy is FRAGILE (<30% of params work) - may be overfit")

print(f"\n3. CYCLE CONSISTENCY:")
print(f"   Does it work in both bull and bear markets?")
print(f"   Check the 'Performance by Market Cycle' section above.")

print(f"\n4. PRACTICAL CONSIDERATIONS:")
print(f"   Trades: {int(best['num_trades'])}")
print(f"   Win Rate: {best['win_rate']:.1f}%")
print(f"   Exit signals: {int(best['exit_signals'])}")

if best['num_trades'] < 5:
    print(f"   ⚠️  Very few trades - limited statistical significance")
elif best['num_trades'] < 10:
    print(f"   ✓ Moderate number of trades")
else:
    print(f"   ✅ Good sample size of trades")

print(f"\n5. RECOMMENDATION:")

if improvement > 500 and better_than_never / total_combos > 0.5 and best['sharpe'] > 0.7:
    print(f"   🎯 PRODUCTION READY!")
    print(f"   ✅ Massive improvement over baseline")
    print(f"   ✅ Robust across parameter variations")
    print(f"   ✅ Strong risk-adjusted returns")
    print(f"\n   💡 Next steps:")
    print(f"      1. Paper trade this strategy live")
    print(f"      2. Monitor MVRV and {best_ma}-day MA daily")
    print(f"      3. Exit when BOTH conditions met")
    print(f"      4. Re-enter on next Buy The Dip signal")
elif improvement > 200 and better_than_never / total_combos > 0.3:
    print(f"   ✓ PROMISING - Continue testing")
    print(f"   Good improvement but validate further")
    print(f"   Consider walk-forward analysis")
else:
    print(f"   ⚠️  INCONCLUSIVE")
    print(f"   Improvement exists but may not be robust")
    print(f"   Consider other approaches")

print(f"\n6. KEY INSIGHTS:")
print(f"   💡 Valuation alone doesn't work (exits too early)")
print(f"   💡 Momentum alone might be too reactive")
print(f"   💡 Combining both captures true trend reversals")
print(f"   💡 MVRV>2.0 + Price<50MA = Sweet spot")

print("\n" + "="*90)

## Summary

This notebook provides a comprehensive analysis of the winning strategy discovered in notebook 82.

**What we tested:**
1. Parameter optimization (MVRV thresholds 1.5-3.0, MA periods 20-100)
2. Trade-by-trade analysis (when exits fired, forward returns)
3. Historical validation (2013-2026 across all cycles)
4. Risk metrics (drawdowns, Sharpe, win rate)
5. Robustness checks (parameter sensitivity)

**Key takeaway:**
After 83 notebooks and testing 80+ exit strategies, we found that **MVRV>2.0 + Price<50MA** is the first strategy to significantly beat "Never Exit" baseline.

The combination of valuation (MVRV) and momentum (price vs MA) captures the essence of good exits:
- Don't exit just because it's expensive (2023-2026 kept going)
- Exit when expensive AND trend breaks (2017/2021 tops)